# 05 — Multilingual contrastive-learning pilot

This experiment trains **multilingual-e5-small** with explicit positive/negative record groups, then compares the same XGBoost configuration with and without embedding cosine similarity. It is inspired by contrastive blocking research, **not a reproduction of SC-Block++**. It does not yet build dense candidates or predict the full test set.

Use the **Python (ml-challenge-neural)** kernel. Run the setup script below once from a terminal if this kernel is absent:

```bash
conda activate ml-challenge-er
python submission/code/business_entity_resolution/src/setup_neural.py
```

Then refresh Jupyter and select the new kernel. The setup inherits the existing pinned environment without changing it and adds PyTorch/Transformers in `.venv-neural`. First setup downloads several GB of CUDA libraries. The model download is about 471 MB and uses a pinned revision; no business identity lookup is performed.

**Run All trains the pilot.** Defaults: up to 10,000 training references, 1,000 tuning references and 1,000 fresh audit references, one neural epoch, length 192 tokens. Training and encoding print measured rates/ETAs. Full runtime is not known until measured on your GPU. Original scripts are retained; transliteration is an additional view. Text beyond field/token limits is truncated.

In [1]:
import os, sys, json, subprocess
from pathlib import Path
os.environ.setdefault('POLARS_MAX_THREADS','8')
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
ancestors=(Path.cwd(),*Path.cwd().parents)
SRC=next(p/r for p in ancestors for r in ('.','src','submission/code/business_entity_resolution/src','code/business_entity_resolution/src') if (p/r/'neural_pilot.py').exists()).resolve()
sys.path.insert(0,str(SRC))
import torch, transformers, polars as pl, numpy as np
from neural_pilot import prepare,train_encoder,encode_records,compare,MODEL_ID,MODEL_REVISION
from train_matcher import atomic_json
from importlib.metadata import version
assert torch.__version__.split('+')[0]=='2.6.0'
assert transformers.__version__=='4.51.3'
assert pl.__version__=='1.44.2'
assert torch.cuda.is_available(), 'Select the neural kernel and verify NVIDIA driver access.'
print('Python:',sys.executable)
print('GPU:',torch.cuda.get_device_name(0))
print('VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,1))
print('Model:',MODEL_ID,MODEL_REVISION)


/home/aline/Projects/ML-Challenge/.venv-neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: /home/aline/Projects/ML-Challenge/.venv-neural/bin/python
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM GiB: 7.7
Model: intfloat/multilingual-e5-small fd1525a9fd15316a2d503bf26ab031a61d056e98


In [2]:
CACHE_ROOT=Path(os.environ['ER_CACHE_ROOT']).expanduser().resolve() if os.environ.get('ER_CACHE_ROOT') else next(p/'cache/normalized_v1' for p in ancestors if (p/'cache/normalized_v1').is_dir())
CANDIDATE_ROOT=Path(os.environ.get('ER_CANDIDATE_ROOT',str(CACHE_ROOT.parent/'candidates_v1'))).expanduser().resolve()
NEURAL_ROOT=Path(os.environ.get('ER_NEURAL_ROOT',str(CACHE_ROOT.parent/'neural_v1'))).expanduser().resolve()
NEURAL_ROOT.mkdir(parents=True,exist_ok=True)
os.environ['HF_HOME']=os.environ.get('ER_HF_HOME',str(CACHE_ROOT.parent/'neural_models'))
TRAIN_ANCHORS=10000
TUNE_ANCHORS=1000
AUDIT_ANCHORS=1000
EPOCHS=1
TRAIN_BATCH=4
ENCODE_BATCH=64
MAX_LENGTH=192
# Configure a NEW ER_NEURAL_ROOT before changing experiment settings after a run.
assert (CANDIDATE_ROOT/'train_fast/stats.json').exists(), 'Complete notebook 04 first.'
atomic_json(NEURAL_ROOT/'environment_versions.json',{k:version(k) for k in ('torch','transformers','safetensors','polars','numpy','xgboost')})
(NEURAL_ROOT/'pip-freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('Experiment cache:',NEURAL_ROOT)


Experiment cache: /home/aline/Projects/ML-Challenge/cache/neural_v1


## Prepare fresh splits and candidates

Training uses the existing training partition. Tune/audit use reserved references that were **not in 02's inspected 5,000-reference pilot**, with separate deterministic hashes. Anchors sharing a known positive target across roles are removed, prioritizing audit, then tune. This can reduce the requested counts. France has no labels here and is not evaluated.

Candidate retrieval is label-blind and uses the complete original training target index. True matches missed by retrieval are **not injected into evaluation candidates**. All training positives may teach the encoder. Its negatives are retrieved nonmatches, excluding every known positive for that anchor and held-out positive targets. Other records in a training batch are not implicitly treated as negatives.

Completed preparation is reused. An interruption during preparation reruns retrieval; lexical feature chunks are resumable. Encoder optimization checkpoints every 100 steps; encoding checkpoints every 4,096 records. Changed data/code/configuration requires a new experiment directory.

In [3]:
data=prepare(CACHE_ROOT,CANDIDATE_ROOT,NEURAL_ROOT,TRAIN_ANCHORS,TUNE_ANCHORS,AUDIT_ANCHORS)
display(data['roles'].group_by('role').len())
display(data['pairs'].group_by('role').agg(pl.len().alias('pairs'),pl.col('label').sum().alias('retrieved_matches')))
print('Unique records to encode:',data['records'].height)


Retrieved 250/12,000 anchors
Retrieved 2,250/12,000 anchors
Retrieved 4,250/12,000 anchors
Retrieved 6,250/12,000 anchors
Retrieved 8,250/12,000 anchors
Retrieved 10,250/12,000 anchors
Features: 25,000/577,678 pairs
Features: 50,000/577,678 pairs
Features: 75,000/577,678 pairs
Features: 100,000/577,678 pairs
Features: 125,000/577,678 pairs
Features: 150,000/577,678 pairs
Features: 175,000/577,678 pairs
Features: 200,000/577,678 pairs
Features: 225,000/577,678 pairs
Features: 250,000/577,678 pairs
Features: 275,000/577,678 pairs
Features: 300,000/577,678 pairs
Features: 325,000/577,678 pairs
Features: 350,000/577,678 pairs
Features: 375,000/577,678 pairs
Features: 400,000/577,678 pairs
Features: 425,000/577,678 pairs
Features: 450,000/577,678 pairs
Features: 475,000/577,678 pairs
Features: 500,000/577,678 pairs
Features: 525,000/577,678 pairs
Features: 550,000/577,678 pairs
Features: 575,000/577,678 pairs
Features: 577,678/577,678 pairs


role,len
str,u32
"""train""",10000
"""audit""",1000
"""tune""",1000


role,pairs,retrieved_matches
str,u32,i64
"""train""",481415,33123
"""tune""",48110,3330
"""audit""",48153,3285


Unique records to encode: 503346


## Train the shared multilingual encoder on GPU

The loss compares each anchor's verified positive with up to three retrieved hard negatives. Up to three positives per training anchor are used. Singletons remain in classifier evaluation but do not provide positive contrastive examples. One fixed epoch avoids selecting the encoder using the audit. GPU mixed precision and gradient checkpointing reduce VRAM usage.

If CUDA runs out of memory, restart the kernel, lower batch sizes and choose a new experiment directory. Keep the completed baseline submission available. Model weights are fetched from the pinned official Hugging Face repository once; saved weights use safetensors.

In [4]:
encoder_path=train_encoder(data,NEURAL_ROOT,epochs=EPOCHS,batch_size=TRAIN_BATCH,max_length=MAX_LENGTH)
print('Saved encoder:',encoder_path)


Encoder 1/6390, loss 1.2312, ETA 0.85 h
Encoder 25/6390, loss 0.1587, ETA 0.18 h
Encoder 50/6390, loss 0.3733, ETA 0.17 h
Encoder 75/6390, loss 0.1093, ETA 0.16 h
Encoder 100/6390, loss 0.0687, ETA 0.16 h
Encoder 125/6390, loss 0.5923, ETA 0.17 h
Encoder 150/6390, loss 0.2380, ETA 0.17 h
Encoder 175/6390, loss 0.0723, ETA 0.17 h
Encoder 200/6390, loss 0.1354, ETA 0.16 h
Encoder 225/6390, loss 0.0432, ETA 0.17 h
Encoder 250/6390, loss 0.0165, ETA 0.17 h
Encoder 275/6390, loss 0.1600, ETA 0.16 h
Encoder 300/6390, loss 0.0261, ETA 0.16 h
Encoder 325/6390, loss 0.7143, ETA 0.18 h
Encoder 350/6390, loss 0.0122, ETA 0.18 h
Encoder 375/6390, loss 0.3565, ETA 0.17 h
Encoder 400/6390, loss 0.0986, ETA 0.17 h
Encoder 425/6390, loss 0.3766, ETA 0.17 h
Encoder 450/6390, loss 0.0008, ETA 0.17 h
Encoder 475/6390, loss 0.5944, ETA 0.17 h
Encoder 500/6390, loss 0.0871, ETA 0.17 h
Encoder 525/6390, loss 0.0434, ETA 0.17 h
Encoder 550/6390, loss 0.0007, ETA 0.17 h
Encoder 575/6390, loss 0.0828, ETA 0.17

## Encode pilot records once

Records shared by several candidate pairs are encoded only once. Embeddings are cached outside the submission. This stage may take longer than fine-tuning because it covers all retrieved candidates, not only the sampled training negatives.

In [5]:
vectors=encode_records(data,encoder_path,NEURAL_ROOT,batch_size=ENCODE_BATCH,max_length=MAX_LENGTH)
print('Embedding matrix:',vectors.shape)


Encoded 4,096/503,346; 2195.7 records/s
Encoded 8,192/503,346; 2254.9 records/s
Encoded 12,288/503,346; 2266.2 records/s
Encoded 16,384/503,346; 2241.4 records/s
Encoded 20,480/503,346; 2232.5 records/s
Encoded 24,576/503,346; 2222.5 records/s
Encoded 28,672/503,346; 2214.5 records/s
Encoded 32,768/503,346; 2206.4 records/s
Encoded 36,864/503,346; 2203.1 records/s
Encoded 40,960/503,346; 2204.2 records/s
Encoded 45,056/503,346; 2200.4 records/s
Encoded 49,152/503,346; 2200.5 records/s
Encoded 53,248/503,346; 2197.4 records/s
Encoded 57,344/503,346; 2193.2 records/s
Encoded 61,440/503,346; 2189.5 records/s
Encoded 65,536/503,346; 2188.9 records/s
Encoded 69,632/503,346; 2189.8 records/s
Encoded 73,728/503,346; 2188.0 records/s
Encoded 77,824/503,346; 2188.2 records/s
Encoded 81,920/503,346; 2188.8 records/s
Encoded 86,016/503,346; 2185.5 records/s
Encoded 90,112/503,346; 2184.9 records/s
Encoded 94,208/503,346; 2185.5 records/s
Encoded 98,304/503,346; 2183.0 records/s
Encoded 102,400/50

## Controlled comparison

Both classifiers use identical train/tune/audit references and candidates. The control has 43 lexical features; the experiment adds one learned cosine-similarity feature. This isolates the benefit of the trained representation from increasing the training sample.

Early stopping and threshold selection use tune only. The final table reports per-reference macro F0.5 plus pair precision/recall and saves per-reference errors. Missing candidates and correct singleton predictions count in the metric. The previous **0.9237** came from a different audit; compare the two rows below directly rather than treating that older number as a paired comparison.

Once viewed, this audit is no longer an untouched final holdout. A gain does not automatically replace the submitted model. Full dense retrieval, its recall evaluation and a new full test run are later steps.

In [6]:
results=compare(data,vectors,NEURAL_ROOT)
display(results)
print('Saved report:',NEURAL_ROOT/'comparison.json')
print('Existing submission outputs have not been replaced.')


[0]	validation_0-logloss:0.21460
[100]	validation_0-logloss:0.03116
[200]	validation_0-logloss:0.02705
[300]	validation_0-logloss:0.02553
[400]	validation_0-logloss:0.02470
[500]	validation_0-logloss:0.02422
[600]	validation_0-logloss:0.02389
[700]	validation_0-logloss:0.02372
[800]	validation_0-logloss:0.02357
[900]	validation_0-logloss:0.02351
[1000]	validation_0-logloss:0.02346
[1100]	validation_0-logloss:0.02344
[1199]	validation_0-logloss:0.02342


/home/aline/.conda/envs/ml-challenge-er/lib/python3.11/site-packages/xgboost/core.py:569: UserWarning: [06:31:33] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1789510313506/work/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[0]	validation_0-logloss:0.21012
[100]	validation_0-logloss:0.02151
[200]	validation_0-logloss:0.01920
[300]	validation_0-logloss:0.01847
[400]	validation_0-logloss:0.01817
[500]	validation_0-logloss:0.01809
[600]	validation_0-logloss:0.01799
[700]	validation_0-logloss:0.01792
[754]	validation_0-logloss:0.01794


model,threshold,audit_macro_f05,pair_precision,pair_recall,best_iteration,audit_anchors,training_anchors
str,f64,f64,f64,f64,i64,i64,i64
"""lexical_control""",0.58,0.920932,0.96307,0.865762,1171,1000,10000
"""lexical_plus_e5""",0.76,0.938389,0.979355,0.876443,694,1000,10000


Saved report: /home/aline/Projects/ML-Challenge/cache/neural_v1/comparison.json
Existing submission outputs have not been replaced.


References: [E5 model card and MIT license](https://huggingface.co/intfloat/multilingual-e5-small), [SC-Block](https://arxiv.org/abs/2303.03132). The local model is below the challenge's 8B limit. The final selected neural package would need its encoder weights, tokenizer, pinned neural dependencies and updated inference code; this pilot alone is not a replacement submission.